In [1]:
import json, glob
from pathlib import Path
import pandas as pd

records = []
for f in sorted(Path("../data/models").glob("*/*/*.json")):
    if f.name == "manifest.json":
        continue
    rec = json.load(open(f))
    rec["model_dir"] = f.parts[-3]   # e.g. gemma-3n-e4b-it
    rec["energy_mode"] = f.parts[-2] # objective_energy / subjective_energy
    rec["file"] = f.name
    records.append(rec)

df = pd.json_normalize(records)

parts = df["file"].str.extract(r"(?P<question>.+)__(?P<graph>[^_]+)__rep(?P<rep>\d+)\.json")
df = df.join(parts)
df["replica"] = df["rep"].astype(int)

df["model"] = df["meta.model"]

# load them and add to the df here
obj = pd.concat([
    pd.read_json("../data/obj/train.jsonl", lines=True),
    pd.read_json("../data/obj/test.jsonl", lines=True),
])
ground_truth = obj.set_index("qid")["answer"]              # "A" or "B"
lean = json.load(open("../data/subj/lean.json"))["lean"]   # qid -> left/right/ambiguous

df["ground_truth"] = df["question"].map(ground_truth)      # NaN on subjective rows
df["political_lean"] = df["question"].map(lean)            # NaN on objective rows

# sanity check: every row got exactly one of the two labels
assert (df["ground_truth"].notna() ^ df["political_lean"].notna()).all()
cols = ['model', 'mode', 'statement', 'J', 'spins_history', 'spins_raw_history',
        'replica', 'ground_truth', 'political_lean']
df[cols].to_json("../data/clean_runs.json", orient="records")

from datasets import Dataset
ds = Dataset.from_pandas(df[cols].reset_index(drop=True))
ds.push_to_hub("physics-of-agents/agent-opinions", private=False)

CommitInfo(commit_url='https://huggingface.co/datasets/physics-of-agents/agent-opinions/commit/15eff2c56c110316b2c2b8e91337c18a705ecabf', commit_message='Upload dataset', commit_description='', oid='15eff2c56c110316b2c2b8e91337c18a705ecabf', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/physics-of-agents/agent-opinions', endpoint='https://huggingface.co', repo_type='dataset', repo_id='physics-of-agents/agent-opinions'), pr_revision=None, pr_num=None)